# Episode 14: Batching: serving a GPT to many users at once

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep014-serving.ipynb)

We serve the model as an API in the shape of OpenAI's completions, measure it against a crowd of players from a game, write the stories for the requests that arrive together as a batch, and run it in Docker, with vLLM as the alternative.

[The video](https://youtu.be/6XGhd_83uwU) · [The episode's page](https://mvgpt.nenuadrian.com/ep014-serving/) · [The docs](https://nenuadrian.github.io/mvGPT/serving/) · [The code at `ep014/end`](https://github.com/nenuadrian/mvGPT/tree/ep014/end)

This notebook runs the episode's commands on its code, as the episode left it. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep014/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")


import shlex
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request


def start(command, url):
    # A server in the background, once it answers at url; its output goes to /tmp/server-<port>.log.
    log = open(f"/tmp/server-{urllib.parse.urlparse(url).port}.log", "w")
    server = subprocess.Popen(shlex.split(command), stdout=log, stderr=subprocess.STDOUT)
    while True:
        try:
            urllib.request.urlopen(url, timeout=1)
            return server
        except urllib.error.HTTPError:
            return server
        except OSError:
            if server.poll() is not None:
                raise RuntimeError(open(log.name).read())
            time.sleep(1)


def stop(server):
    server.terminate()
    server.wait()

## The server
An OpenAI-style completions API, with the model from the Hub. It runs in the background, and
the next cells talk to it.

In [ ]:
server = start("python server.py", "http://127.0.0.1:8000/health")

In [ ]:
!curl -s http://127.0.0.1:8000/v1/completions -H 'Content-Type: application/json' -d '{"prompt": "Words: dragon, castle, brave\nStory:", "max_tokens": 60}' | python -m json.tool

## A crowd of players
The episode's first server wrote one request at a time; `--batch 1` is that server again.

In [ ]:
stop(server)
server = start("python server.py --batch 1", "http://127.0.0.1:8000/health")
!python load.py --out results/load-one.json

## Batching
The tests, then the crowd again, against batches of up to 16, and of up to 64.

In [ ]:
!python -m pytest tests/test_server.py -v

In [ ]:
stop(server)
server = start("python server.py", "http://127.0.0.1:8000/health")
!python load.py --out results/load-batch16.json
stop(server)
server = start("python server.py --batch 64", "http://127.0.0.1:8000/health")
!python load.py --out results/load-batch64.json
stop(server)

## In Docker
The episode builds the server's image with `docker build --target server -t mvgpt-server .`
and runs it: Colab can't run Docker, so that part isn't here.

Next: [episode 15, Text diffusion: a GPT that writes all at once](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep015-diffusion.ipynb)